# តម្រង់ម៉ូឌែលជាមួយ Microsoft Foundry

កំណត់ត្រានេះបញ្ចប់នឹង [ចរន្តការងារតម្រង់ម៉ូឌែល Microsoft Foundry](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning?WT.mc_id=academic-105485-koreyst) ។ វាប្រើប្រាស់ **OpenAI Python SDK (v1)** ដែលបញ្ជូនទៅចំណុចបញ្ចប់ `/openai/v1/` របស់ធនធាន Foundry របស់អ្នក ដូច្នេះកូដដូចគ្នានេះក៏ដំណើរការជាមួយវេទិការបស់ OpenAI ដោយប្ដូរតែការតម្លើងអតិថិជនប៉ុណ្ណោះ។

> **លក្ខខណ្ឌជាមុន**
> - គម្រោង [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) មួយ ជាមួយតួនាទី **Foundry Owner** (ចាំបាច់សម្រាប់ធ្វើ fine-tune និងដាក់ឲ្យដំណើរការ)។
> - `pip install "openai>=1.0" python-dotenv`
> - ឯកសារ `.env` មាន `AZURE_OPENAI_ENDPOINT` និង `AZURE_OPENAI_API_KEY` (មើលការណែនាំសម្រាប់កំណត់ជម្រៅ [course setup guide](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst))។
> - ម៉ូឌែលមូលដ្ឋានដែលគាំទ្របច្ចុប្បន្ន ដូចជា `gpt-4.1-mini` (មើលបញ្ជីម៉ូឌែលដែលអាច fine-tune បាន [fine-tunable models list](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models))។

ការតម្រង់ (Fine-tuning) បង្កើនគុណភាពម៉ូឌែលមូលដ្ឋានដោយបណ្តុះបណ្តាលវា ម្តងទៀតលើឧទាហរណ៍បន្ថែមដែលពាក់ព័ន្ធនឹងភារកិច្ចរបស់អ្នក។ បច្ចេកទេស prompt-engineering ដូចជា _few-shot learning_ និង _retrieval-augmented generation_ ធ្វើឲ្យ prompt មានទិន្នន័យពាក់ព័ន្ធ ប៉ុន្តែវាមានកំណត់ដោយបង្អួច context window របស់ម៉ូឌែល។

ជាមួយការតម្រង់ អ្នកបណ្តុះបណ្តាលម៉ូឌែលផ្ទាល់ (ប្រើឧទាហរណ៍ច្រើនជាង context window) ហើយដាក់ជាផលិតផល _ផ្ទាល់ខ្លួន_ ដែលមិនត្រូវការឧទាហរណ៍ទាំងនោះនៅពេលប្រើប្រាស់។ វាជួយបង្កើនគុណភាព ផ្លាស់ទីបញ្ជី context window និងអាចបន្ដដុះថ្លៃ និងពេលឆ្លើយដោយកាត់បន្ថយពាក្យហៅ។ នៅខាងក្រោមមុខងារ Foundry ប្រើ **LoRA (low-rank adaptation)** សម្រាប់ការបណ្តុះបណ្តាលប្រសិទ្ធិភាព។

Foundry គាំទ្របច្ចេកទេសបីប្រភេទ៖ **supervised fine-tuning (SFT)** - ដែលប្រើសម្រាប់មេរៀននេះ - រួមមាន **DPO** (ការតម្រង់ចំណង់ចំណូលចិត្ត) និង **RFT** (ការតម្រង់ជាមធ្យោបាយបម្រាស់)។ ជួរនីតិសម្រាប់ការងារមានចំនួនបួនជំហាន៖

1. រៀបចំ និងផ្ទុកទិន្នន័យបណ្តុះបណ្តាល **និង** ផ្ទៀងផ្ទាត់។
2. បើករូបការបណ្តុះបណ្តាលដើម្បីបង្កើតម៉ូឌែល fine-tuned។
3. តាមដានការងារ ពិនិត្យមេទ្រីក និងជ្រើសយក checkpoint។
4. ដាក់បញ្ចេញម៉ូឌែល fine-tuned ហើយប្រើវាសម្រាប់ការវិភាគ។

ក្នុងមេរៀននេះ យើងតម្រង់ `gpt-4.1-mini` ជាមួយ SFT ដើម្បីបង្កើត chatbot ដែលឆ្លើយសំនួរអំពីតារាង périodic ជាមួយ limerick។

---


### ជំហាន 1.1៖ រៀបចំទិន្នន័យរបស់អ្នក

ចូរយើងបង្កើត chatbot មួយដែលជួយអ្នកយល់ពីតារាងបូករួមធាតុដោយការឆ្លើយសំណួរអំពីធាតុជាមួយ limerick ។ ក្នុងមេរៀនសាមញ្ញ _នេះ_ យើងនឹងបង្កើតទិន្នន័យសម្រាប់បណ្តុះបណ្តាលម៉ូដែលជាមួយឧទាហរណ៍ព្យួរមួយចំនួននៃចម្លើយដែលបង្ហាញទ្រង់ទ្រាយដែលខ្ញុំរំពឹងទុក។ ក្នុងករណីប្រើប្រាស់ពិត អ្នកនឹងត្រូវបង្កើតទិន្នន័យជាមួយឧទាហរណ៍ច្រើនជាងនេះ។ អ្នកអាចប្រើទិន្នន័យបើកសម្រាប់ដែនប្រើប្រាស់របស់អ្នកបើមាន ហើយធ្វើទ្រង់ទ្រាយឡើងវិញសម្រាប់ការបណ្តុះបណ្តាលលម្អិត។

ពីព្រោះយើងផ្តោតលើ `gpt-4.1-mini` និងស្វែងរកចម្លើយមួយជុំដំណើរ (បញ្ចប់ការជជែក) យើងអាចបង្កើតឧទាហរណ៍ដោយប្រើ [ទ្រង់ទ្រាយដែលបានផ្តល់](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst) ដែលបង្ហាញតម្រូវការបញ្ចប់ការជជែករបស់ OpenAI ។ ប្រសិនបើអ្នករំពឹងថាត្រូវមានការជជែកច្រើនជុំ អ្នកនឹងប្រើ [ទ្រង់ទ្រាយឧទាហរណ៍ជជែកច្រើនជុំ](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst) ដែលរួមបញ្ចូលជាមួយប៉ារ៉ាម៉ែត្រ `weight` ដើម្បីព្យាបាលថាសារណាអ្វីគួរត្រូវបានប្រើ (ឬមិន) ក្នុងដំណើរការបណ្តុះបណ្តាលលម្អិត។

យើងនឹងប្រើទ្រង់ទ្រាយមួយជុំតែមួយសាមញ្ញសម្រាប់មេរៀននេះ។ ទិន្នន័យមានទ្រង់ទ្រាយ [jsonl](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst) ជាមួយកំណត់ត្រា ១ នៅក្នុងមួយជួរដេក ខុសគ្នាជារូបមន្ដ JSON ។ ការចម្លងខាងក្រោមបង្ហាញពីកំណត់ត្រា ២ ជាឧទាហរណ៍ - មើល [training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl) សម្រាប់ឧទាហរណ៍ពេញលេញ (១០ ឧទាហរណ៍) ដើម្បីប្រើសម្រាប់មេរៀនបណ្តុះបណ្ដាលលម្អិតរបស់យើង។ **ចំណាំ៖** កំណត់ត្រាតម្រូវត្រូវបានកំណត់ក្នុងជួរដេកតែមួយ (មិនត្រូវបំបែកជាជួរដេកខុសៗគ្នាដូចជា JSON តម្រៀប) 

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

ក្នុងករណីប្រើប្រាស់ពិត អ្នកនឹងត្រូវមានឧទាហរណ៍ច្រើនជាងនេះសម្រាប់លទ្ធផលល្អ - ការជួញដូរនឹងត្រូវបានធ្វើរវាងគុណភាពចម្លើយ និងពេលវេលា/ដំណើរការចំណាយសម្រាប់បណ្តុះបណ្តាលលម្អិត។ យើងកំពុងប្រើឧទាហរណ៍តូច ដូច្នេះអាចបញ្ចប់បណ្តុះបណ្តាលលម្អិតប្រញាប់ៗដើម្បីបង្ហាញដំណើរ។ មើល [ឧទាហរណ៍ OpenAI Cookbook នេះ](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) សម្រាប់មេរៀនបណ្តុះបណ្តាលលម្អិតដែលស្មុគស្មាញជាងនេះ។


---

### ជំហានទី 1.2: ផ្ទុកឡើងឯកសារទិន្នន័យរបស់អ្នក

ផ្ទុកឡើងឯកសារបណ្តុះបណ្តាល និង ឯកសារត្រួតពិនិត្យរបស់អ្នកទៅ Foundry ដោយប្រើ Files API (`purpose="fine-tune"`). ការផ្តល់ឱ្យមាន **ឯកសារត្រួតពិនិត្យ** អនុញ្ញាតឱ្យ Foundry រាយការណ៍ការបាត់បង់ត្រួតពិនិត្យ ដើម្បីឲ្យអ្នកអាចស្គាល់ការតូចតាចរ៉ែននៃទិន្នន័យ។

មុនពេលរត់កូដខាងក្រោម សូមប្រាកដថាអ្នកបាន:
 - ដំឡើង SDK: `pip install "openai>=1.0" python-dotenv`
 - បង្កើតឯកសារ `.env` ជាមួយ `AZURE_OPENAI_ENDPOINT` និង `AZURE_OPENAI_API_KEY`

កូដនេះបង្កើតភ្ញៀវ OpenAI ដែលស្របទៅកាន់ /openai/v1/ ដែនដីនៃធនធាន Foundry របស់អ្នក បន្ទាប់មកផ្ទុកឯកសារទាំងពីរនិងបង្ហាញអត្តសញ្ញាណរបស់ពួកវា។


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Point the OpenAI SDK at your Microsoft Foundry resource's /openai/v1/ endpoint.
# (For the OpenAI platform instead, use: client = OpenAI()  with OPENAI_API_KEY set.)
client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
)

training_response = client.files.create(
    file=open("./training-data.jsonl", "rb"), purpose="fine-tune"
)
validation_response = client.files.create(
    file=open("./validation-data.jsonl", "rb"), purpose="fine-tune"
)

training_file_id = training_response.id
validation_file_id = validation_response.id

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)


---

### ជំហាន 2.1៖ បង្កើតការងារកែសម្រួលតាម SDK


In [ ]:
# Create the fine-tuning job.
# trainingType "GlobalStandard" is the recommended tier (lower cost, faster queue).
# Use "Standard" if you need regional data residency, or "Developer" for cheap experiments.
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini",              # base model to fine-tune
    training_file=training_file_id,
    validation_file=validation_file_id,
    suffix="elements-limerick",        # helps you identify the resulting model
    seed=105,                          # makes the run reproducible
    extra_body={"trainingType": "GlobalStandard"},
)

job_id = job.id
print("Fine-tuning Job ID:", job_id)
print("Status:", job.status)


---

### ជំហាន 2.2: ពិនិត្យស្ថានភាពរបស់ការងារ

នេះគឺជាចំនុចខ្លះៗដែលអ្នកអាចធ្វើជាមួយ API `client.fine_tuning.jobs`៖
- `client.fine_tuning.jobs.list(limit=<n>)` - រាយបញ្ជីការងារ fine-tuning  ចុងក្រោយ n ការងារ
- `client.fine_tuning.jobs.retrieve(<job_id>)` - ទទួលយកព័ត៌មានលម្អិតនៃការងារពិសេសមួយ
- `client.fine_tuning.jobs.cancel(<job_id>)` - បោះបង់ការងារ
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<n>)` - រាយបញ្ជីព្រឹត្តិការណ៍ពីការងារ
- `client.fine_tuning.jobs.checkpoints.list(<job_id>)` - រាយបញ្ជីចំណុចផ្អាក់ដែលអាចផ្សព្វផ្សាយបាន (epoch ចុងក្រោយពីរបី)

នៅពេលការងារចាប់ផ្តើម Foundry ជាមុនសិន _ធ្វើការផ្ទៀងផ្ទាត់ឯកសារបណ្តុះបណ្តាល_ ដើម្បីប្រាកដថាទិន្នន័យមានទ្រង់ទ្រាយត្រឹមត្រូវ។ ការបណ្តុះបណ្តាលបន្ទាប់មកចាប់ផ្តើមរត់ ហើយអាចយកពេលពីមួយនាទីដល់ម៉ោង បានអាស្រ័យទៅលើទំហំនៃម៉ូដែល និងឯកសារទិន្នន័យ។


In [ ]:
# List the last 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the current state of your fine-tuning job
client.fine_tuning.jobs.retrieve(job_id)

# List up to 10 events from the job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)


In [ ]:
# Track the job status until it is complete
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)


---

### ជំហានទី 2.3: តាមដានព្រឹត្តិការណ៍ដើម្បីត្រួតពិនិត្យវឌ្ឍនភាព


In [ ]:
# Track progress in a more granular way by checking events.
# Re-run this cell until you see "The job has successfully completed".
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)


In [ ]:
# When the job finishes, the last few epochs are available as deployable checkpoints.
# Best practice: don't blindly deploy the final epoch - review the checkpoints and pick
# the one that generalizes best (watch train_loss vs. valid_loss and token accuracy).
checkpoints = client.fine_tuning.jobs.checkpoints.list(job_id)
for cp in checkpoints.data:
    print(cp.fine_tuned_model_checkpoint, "| step:", cp.step_number)


### ជំហាន 2.4៖ ពិនិត្យស្ថានភាព, métics, និងចំណុចបញ្ឈប់នៅក្នុងច្រក Foundry


អ្នកក៏អាចតាមដានការងារនៅក្នុង [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) នៅក្រោម **Build > Fine-tuning** ផងដែរ។ ជ្រើសរើសការងាររបស់អ្នកដើម្បីមើលស្ថានភាព វិក្កយបត្របណ្តុះបណ្តាល ប៉ារ៉ាម៉ែត្រអសមរម្យ និងមេត្រីក។ ត្រួតពិនិត្យមេត្រីកទាំងនេះ៖

- `train_loss` និង `full_valid_loss` - គួរត្រូវបានកាត់បន្ថយជាប់ជាមួយពេលវេលា។
- `train_mean_token_accuracy` និង `full_valid_mean_token_accuracy` - គួរត្រូវបានកើនឡើង។

ប្រសិនបើកោងបណ្តុះបណ្តាល និងបញ្ជាក់ទន់ភ្លឺផុតពីគ្នា អ្នកប្រហែលជា overfitting - សូមសាកល្បងកំណត់ស្ទីលភ្លេងកំណត់ឥតច្រើនឬវ៉ាល្យូបតិចជាងសម្រាប់វេហ្កាសិក្សា។ រូបភាពខាងក្រោមបង្ហាញពីប្រភេទស្ថានភាព សារនិងផ្ទាំងមេត្រីកដែលអ្នកនឹងមើលឃើញ (UI ច្បាស់លាស់ខុសគ្នាតាមអ្នកផ្គត់ផ្គង់)។

![Fine-tuning job status](../../../../../translated_images/km/fine-tuned-model-status.563271727bf7bfba.webp)


អ្នកក៏អាចមើលសារស្ថានភាព និងគន្លងវាស់វែងដោយរមូរទៅក្រោមបន្ថែមទៀតក្នុងផ្ទាំងត្រួតពិនិត្យមើលជាភាគច្រើនដូចដែលបានបង្ហាញ៖

| Messages | Metrics |
|:---|:---|
| ![Messages](../../../../../translated_images/km/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/km/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### ជំហាន 3.1៖ ទាញយកអត្តសញ្ញាណម៉ូដែលដែលបានបង្រៀនឡើងវិញ

នៅពេលការងារជោគជ័យ `response.fine_tuned_model` នាំផ្ទុកអត្តសញ្ញាណនៃម៉ូដែលផ្ទាល់ខ្លួនរបស់អ្នក (ឧទាហរណ៍ `gpt-4.1-mini-2025-04-14.ft-...`)។ លើ Azure អ្នកត្រូវ **ប្រើប្រាស់** ម៉ូដែលនោះមុនពេលអ្នកអាចហៅវា។ 


In [ ]:
# Retrieve the id of the fine-tuned model once the job has succeeded
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)


### ជំហាន 3.2៖ ដោះដូរស្តង់ដារដែលបានលាត់តែម្តង

ខុសពីវេទិកា OpenAI (ដែលអ្នកអាចហៅ id នៃម៉ូដែលដែលបានលាត់តែក្នុងផ្ទាល់) Microsoft Foundry ត្រូវការ​ឲ្យអ្នក **ដោះដូរ** ម៉ូដែលជាមុនសិន។ វិធីងាយបំផុតគឺប្រើ [portal Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst): ចូលទៅកាន់ **Build > Fine-tuning**, ជ្រើសរើសការងារដែលបានបញ្ចប់រួចហើយ និងជ្រើស **Deploy**។ ផ្ដល់ឈ្មោះសម្រាប់ការដោះដូរ (ឧ. `elements-limerick`) - ឈ្មោះការដោះដូរនេះជាឈ្មោះដែលអ្នកនឹងហៅពីកូដ។

អ្នកក៏អាចដោះដូរជាកម្មវិធីតាមរយៈ REST/ARM API នៃ control-plane; មើល [មគ្គុទេសក៍ការដោះដូរ](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning-deploy?WT.mc_id=academic-105485-koreyst)។ ចងចាំថា ម៉ូដែលផ្ទាល់ខ្លួនដែលបានដោះដូរនឹងគិតថ្លៃរោមម៉ោង ហើយការដោះដូរដែលអសកម្មនឹងត្រូវបានយកចេញបន្ទាប់ពី ១៥ ថ្ងៃ។


In [ ]:
# Once deployed, call your fine-tuned model by its DEPLOYMENT name via the Responses API.
# (On the OpenAI platform you'd pass fine_tuned_model_id directly instead.)
deployment_name = "elements-limerick"  # the name you gave the deployment in Foundry

completion = client.responses.create(
    model=deployment_name,
    input=[
        {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
        {"role": "user", "content": "Tell me about Strontium"},
    ],
    store=False,
)
print(completion.output_text)


---

### ជំហានទី 3.3: សាកល្បងម៉ូដែលដែលបានបង្រៀនលម្អរបស់អ្នកនៅក្នុងលំហយោល Foundry

អ្នកអាចសាកល្បងម៉ូដែលដែលបានចេញផ្សាយនៅក្នុង [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) **Playground** - ជ្រើសរើសការចេញផ្សាយ fine-tuned របស់អ្នកពីម៉ឺនុយចុះចំណុចម៉ូដែល ហើយព្យាយាមបញ្ចូលប្រធានបទ។ ប្រើ **សារប្រព័ន្ធដដែល** ដែលអ្នកបានបង្រៀនជាមួយ វា ការប្រើសារប្រព័ន្ធផ្សេងគ្នាអាចប្តូរប្រតិបត្តិការនេះបាន។

> **ដំណឹង:** ប្រៀបធៀបម៉ូដែល fine-tuned ជាមួយ `gpt-4.1-mini` មូលដ្ឋានជាប់គ្នា។ សំគាល់ពេលម៉ូដែល fine-tuned ឆ្លើយតបក្នុងទ្រង់ទ្រាយ limerick ពីគំរូរបស់អ្នក ខណៈដែលម៉ូដែលមូលដ្ឋានតែអនុវត្តតាមប្រធានបទប្រព័ន្ធ។

**នេះគឺជាគំរូសាមញ្ញមួយសម្រាប់បង្ហាញដំណើរការ មិនមែនជាដាតាសេតពិតទេ។** នៅក្នុងការផលិត អ្នកនឹងបង្រៀនលម្អលើទិន្នន័យពិត (ឧ. បញ្ជីផលិតផលសម្រាប់សេវាកម្មភ្ញៀវ) ដែលគុណភាពខុសគ្នាច្បាស់ជាងជាងនេះ - ហើយការច្នៃប្រឌិតប្រធានបទតែតែដោយឯងនឹងត្រូវការច្រើនតូខិនសម្រាប់សំណួរ។ សម្រាប់ឧទាហរណ៍លម្អិតជាងនេះ សូមមើល [OpenAI Cookbook fine-tuning guide](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) និង [Foundry fine-tuning tutorial](https://learn.microsoft.com/azure/ai-foundry/openai/tutorials/fine-tune?WT.mc_id=academic-105485-koreyst)។

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
